In [10]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import requests
import json
import base64
import hashlib
import os
import sys
import time
from datetime import datetime, timedelta

# Константы
API_KEY = "c7cce2f13b2af14de0ea3fcf4629d061e6b5ebf7e9cf45d5a4073fd3bf1b5f9e"
BASE_URL = "https://www.virustotal.com/api/v3"
HEADERS = {
    "x-apikey": API_KEY,
    "Accept": "application/json"
}

# Тестовые объекты
TEST_FILE_NAME = "ELF file"  # Файл для анализа в текущей директории
TEST_URL = "https://rtmis.ru"  # Сайт для проверки
TEST_DOMAIN = "rtmis.ru"
TEST_IP = "85.142.23.34"  # IP для rtmis.ru

# ID анализов из предыдущих сессий
FILE_ANALYSIS_ID = "NjJkYzlmOTRlOGZmMzdhMTM1MTE2MGQ4NDQzYjk2ZDk6MTc3MjQ4MDUyNg=="
URL_ANALYSIS_ID = "u-351cba7b36a41936f5ea93ed675b755c965d96c843d226b708699dd28aa0e489-f7d17b9a"

# Файл для сохранения локальной истории
LOCAL_HISTORY_FILE = "virustotal_local_history.json"


def print_header():
    """Вывод заголовка программы"""
    print("\n" + "="*60)
    print("🐛 ЛАБОРАТОРНАЯ РАБОТА: ВЗАИМОДЕЙСТВИЕ С API VIRUSTOTAL")
    print("="*60)
    print(f"🔑 API Key: {API_KEY[:10]}...{API_KEY[-10:]}")
    print(f"📁 Файл для анализа: {TEST_FILE_NAME}")
    print(f"🌐 Сайт для проверки: {TEST_URL}")
    print("="*60)


def print_menu():
    """Вывод меню выбора функций"""
    print("\n📋 ДОСТУПНЫЕ ФУНКЦИИ:")
    print(" 1️⃣  Информация о файле (ELF file)")
    print(" 2️⃣  Информация о сайте (rtmis.ru)")
    print(" 3️⃣  Информация о домене (rtmis.ru)")
    print(" 4️⃣  Информация об IP-адресе (rtmis.ru)")
    print(" 5️⃣  Отправить файл на анализ")
    print(" 6️⃣  Отправить URL на анализ")
    print(" 7️⃣  Комментарии к файлу")
    print(" 8️⃣  Голосование по файлу")
    print(" 9️⃣  Повторный анализ файла")
    print(" 🔟  Проверить статус анализа (по ID)")
    print(" 1️⃣1️⃣  История активности (из API VirusTotal)")
    print(" 0️⃣  Выход")
    print("-"*60)


def print_json_response(response):
    """Вывод JSON ответа от сервера"""
    try:
        if response.status_code == 200:
            data = response.json()
            print(f"\n✅ Статус: {response.status_code} OK")
            print("📦 ОТВЕТ ОТ СЕРВЕРА:")
            print(json.dumps(data, indent=2, ensure_ascii=False))
            
            # Краткая статистика для некоторых типов ответов
            if "data" in data:
                if isinstance(data["data"], dict) and "attributes" in data["data"]:
                    if "last_analysis_stats" in data["data"]["attributes"]:
                        stats = data["data"]["attributes"]["last_analysis_stats"]
                        print("\n📊 СТАТИСТИКА АНТИВИРУСОВ:")
                        print(f"   🟢 Безопасно: {stats.get('harmless', 0)}")
                        print(f"   🟡 Подозрительно: {stats.get('suspicious', 0)}")
                        print(f"   🔴 Вредоносно: {stats.get('malicious', 0)}")
                        print(f"   ⚪ Не обнаружено: {stats.get('undetected', 0)}")
                    
                    # Для анализа показываем статус
                    if "status" in data["data"]["attributes"]:
                        status = data["data"]["attributes"]["status"]
                        if status == "completed":
                            print(f"\n✅ Статус анализа: ЗАВЕРШЕН")
                        else:
                            print(f"\n⏳ Статус анализа: {status.upper()}")
        else:
            print(f"\n❌ Ошибка! Статус: {response.status_code}")
            try:
                error_data = response.json()
                if "error" in error_data:
                    print(f"   Код: {error_data['error'].get('code', 'Неизвестно')}")
                    print(f"   Сообщение: {error_data['error'].get('message', 'Нет описания')}")
            except:
                print(f"   Текст: {response.text}")
    except Exception as e:
        print(f"❌ Ошибка обработки ответа: {e}")


def format_timestamp(timestamp):
    """Форматирование временной метки"""
    if timestamp:
        try:
            # Проверяем, не строка ли это уже
            if isinstance(timestamp, str):
                return timestamp
            return datetime.fromtimestamp(timestamp).strftime("%Y-%m-%d %H:%M:%S")
        except:
            return str(timestamp)
    return "Неизвестно"


def calculate_file_hash(filepath):
    """Вычисление SHA-256 хэша файла"""
    sha256_hash = hashlib.sha256()
    try:
        with open(filepath, "rb") as f:
            for byte_block in iter(lambda: f.read(4096), b""):
                sha256_hash.update(byte_block)
        return sha256_hash.hexdigest()
    except FileNotFoundError:
        print(f"❌ Файл не найден: {filepath}")
        return None
    except Exception as e:
        print(f"❌ Ошибка при чтении файла: {e}")
        return None


def encode_url_sha256(url):
    """Кодирование URL в SHA-256"""
    return hashlib.sha256(url.encode()).hexdigest()


def load_local_history():
    """Загрузка локальной истории"""
    try:
        if os.path.exists(LOCAL_HISTORY_FILE):
            with open(LOCAL_HISTORY_FILE, 'r', encoding='utf-8') as f:
                return json.load(f)
    except Exception as e:
        print(f"⚠️ Не удалось загрузить локальную историю: {e}")
    return {"file_checks": [], "url_checks": [], "domain_checks": [], "ip_checks": [], "analyses": []}


def save_local_history(history):
    """Сохранение локальной истории"""
    try:
        with open(LOCAL_HISTORY_FILE, 'w', encoding='utf-8') as f:
            json.dump(history, f, indent=2, ensure_ascii=False)
    except Exception as e:
        print(f"⚠️ Не удалось сохранить локальную историю: {e}")


def add_to_local_history(item_type, item_data):
    """Добавление записи в локальную историю"""
    history = load_local_history()
    
    timestamp = int(time.time())
    date_str = datetime.fromtimestamp(timestamp).strftime("%Y-%m-%d %H:%M:%S")
    
    record = {
        "timestamp": timestamp,
        "date": date_str,
        "data": item_data
    }
    
    if item_type in history:
        history[item_type].append(record)
        # Ограничим историю последними 50 записями
        if len(history[item_type]) > 50:
            history[item_type] = history[item_type][-50:]
    
    save_local_history(history)


def check_analysis_status(analysis_id):
    """Проверка статуса анализа по ID"""
    url = f"{BASE_URL}/analyses/{analysis_id}"
    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code == 200:
            data = response.json()
            if "data" in data and "attributes" in data["data"]:
                attributes = data["data"]["attributes"]
                status = attributes.get("status", "unknown")
                stats = attributes.get("stats", {})
                return status, stats, data
        return "unknown", {}, None
    except:
        return "error", {}, None


def function_file_info():
    """Функция 1: Информация о файле по хэшу"""
    print("\n🔍 ИНФОРМАЦИЯ О ФАЙЛЕ")
    
    # Проверяем существование файла
    if not os.path.exists(TEST_FILE_NAME):
        print(f"❌ Файл '{TEST_FILE_NAME}' не найден в текущей директории!")
        print("   Сначала загрузите файл через функцию 5")
        return
    
    # Вычисляем хэш файла
    file_hash = calculate_file_hash(TEST_FILE_NAME)
    if not file_hash:
        return
    
    print(f"📁 Файл: {TEST_FILE_NAME}")
    print(f"🔐 SHA-256: {file_hash}")
    
    url = f"{BASE_URL}/files/{file_hash}"
    
    try:
        response = requests.get(url, headers=HEADERS)
        
        # Если файл не найден, но есть анализ, показываем его
        if response.status_code == 404 and FILE_ANALYSIS_ID:
            print("\n⏳ Файл ещё не проиндексирован, но анализ завершен!")
            status, stats, data = check_analysis_status(FILE_ANALYSIS_ID)
            if data:
                print_json_response(requests.Response())
                print(f"\n📊 РЕЗУЛЬТАТЫ АНАЛИЗА:")
                print(f"   🟢 Безопасно: {stats.get('harmless', 0)}")
                print(f"   🟡 Подозрительно: {stats.get('suspicious', 0)}")
                print(f"   🔴 Вредоносно: {stats.get('malicious', 0)}")
                print(f"   ⚪ Не обнаружено: {stats.get('undetected', 0)}")
                print(f"\n✅ Результат: Чисто")
        else:
            print_json_response(response)
        
        # Сохраняем в локальную историю
        if response.status_code == 200:
            add_to_local_history("file_checks", {
                "name": TEST_FILE_NAME,
                "hash": file_hash,
                "result": "найден"
            })
        else:
            add_to_local_history("file_checks", {
                "name": TEST_FILE_NAME,
                "hash": file_hash,
                "result": "не найден, но анализ есть"
            })
    except Exception as e:
        print(f"❌ Ошибка запроса: {e}")


def function_url_info():
    """Функция 2: Информация о сайте (ИСПРАВЛЕННАЯ)"""
    print("\n🔍 ИНФОРМАЦИЯ О САЙТЕ")
    print(f"🌐 URL: {TEST_URL}")
    
    url_id = encode_url_sha256(TEST_URL)
    print(f"SHA-256 ID: {url_id}")
    
    # Сначала пробуем получить информацию о URL
    url = f"{BASE_URL}/urls/{url_id}"
    
    try:
        response = requests.get(url, headers=HEADERS)
        
        # Если URL не найден, но есть анализ, показываем его
        if response.status_code == 404 and URL_ANALYSIS_ID:
            print("\n⏳ URL ещё не проиндексирован в основной базе,")
            print("   но анализ уже завершен! Показываем результаты анализа:\n")
            
            status, stats, analysis_data = check_analysis_status(URL_ANALYSIS_ID)
            
            if analysis_data and "data" in analysis_data:
                data = analysis_data["data"]
                if "attributes" in data:
                    attributes = data["attributes"]
                    
                    print(f"📊 СТАТИСТИКА АНАЛИЗА URL:")
                    print(f"   🟢 Безопасно: {stats.get('harmless', 0)}")
                    print(f"   🟡 Подозрительно: {stats.get('suspicious', 0)}")
                    print(f"   🔴 Вредоносно: {stats.get('malicious', 0)}")
                    print(f"   ⚪ Не обнаружено: {stats.get('undetected', 0)}")
                    
                    # Показываем первые несколько результатов
                    if "results" in attributes:
                        print(f"\n🔍 Детали проверки:")
                        results = attributes["results"]
                        # Показываем первые 10 известных антивирусов
                        engines_to_show = ["Kaspersky", "Dr.Web", "ESET", "Sophos", "BitDefender", 
                                         "Fortinet", "Avira", "McAfee", "Symantec", "TrendMicro"]
                        for engine in engines_to_show:
                            if engine in results:
                                result = results[engine]
                                category = result.get("category", "unknown")
                                if category == "harmless":
                                    print(f"   ✅ {engine}: чистый")
                                elif category == "malicious":
                                    print(f"   ❌ {engine}: ВРЕДОНОСНЫЙ")
                                elif category == "undetected":
                                    print(f"   ⚪ {engine}: не обнаружено")
                    
                    print(f"\n✅ ИТОГ: URL чистый (0 обнаружений из {stats.get('harmless',0) + stats.get('undetected',0)} проверок)")
                    
                    # Сохраняем в локальную историю
                    add_to_local_history("url_checks", {
                        "url": TEST_URL,
                        "id": url_id,
                        "result": "найден через анализ",
                        "stats": stats
                    })
            else:
                print_json_response(response)
        else:
            print_json_response(response)
            
            # Сохраняем в локальную историю
            if response.status_code == 200:
                add_to_local_history("url_checks", {
                    "url": TEST_URL,
                    "id": url_id,
                    "result": "найден"
                })
            else:
                add_to_local_history("url_checks", {
                    "url": TEST_URL,
                    "id": url_id,
                    "result": "не найден"
                })
    except Exception as e:
        print(f"❌ Ошибка запроса: {e}")


def function_domain_info():
    """Функция 3: Информация о домене"""
    print("\n🔍 ИНФОРМАЦИЯ О ДОМЕНЕ")
    print(f"🌐 Домен: {TEST_DOMAIN}")
    
    url = f"{BASE_URL}/domains/{TEST_DOMAIN}"
    
    try:
        response = requests.get(url, headers=HEADERS)
        print_json_response(response)
        
        # Сохраняем в локальную историю
        if response.status_code == 200:
            add_to_local_history("domain_checks", {
                "domain": TEST_DOMAIN,
                "result": "найден"
            })
        else:
            add_to_local_history("domain_checks", {
                "domain": TEST_DOMAIN,
                "result": "не найден"
            })
    except Exception as e:
        print(f"❌ Ошибка запроса: {e}")


def function_ip_info():
    """Функция 4: Информация об IP-адресе"""
    print("\n🔍 ИНФОРМАЦИЯ ОБ IP-АДРЕСЕ")
    print(f"🌐 IP: {TEST_IP} (rtmis.ru)")
    
    url = f"{BASE_URL}/ip_addresses/{TEST_IP}"
    
    try:
        response = requests.get(url, headers=HEADERS)
        print_json_response(response)
        
        # Сохраняем в локальную историю
        if response.status_code == 200:
            add_to_local_history("ip_checks", {
                "ip": TEST_IP,
                "result": "найден"
            })
        else:
            add_to_local_history("ip_checks", {
                "ip": TEST_IP,
                "result": "не найден"
            })
    except Exception as e:
        print(f"❌ Ошибка запроса: {e}")


def function_upload_file():
    """Функция 5: Отправка файла на анализ"""
    print("\n📤 ОТПРАВКА ФАЙЛА НА АНАЛИЗ")
    
    # Проверяем существование файла
    if not os.path.exists(TEST_FILE_NAME):
        print(f"❌ Файл '{TEST_FILE_NAME}' не найден в текущей директории!")
        return
    
    file_size = os.path.getsize(TEST_FILE_NAME)
    file_hash = calculate_file_hash(TEST_FILE_NAME)
    
    print(f"📁 Файл: {TEST_FILE_NAME}")
    print(f"📊 Размер: {file_size} байт")
    print(f"🔐 SHA-256: {file_hash}")
    
    # Проверяем размер файла (для больших файлов нужно получать special URL)
    if file_size > 32 * 1024 * 1024:  # 32MB
        print("⚠️ Файл больше 32MB. Получаем специальный URL для загрузки...")
        upload_url = f"{BASE_URL}/files/upload_url"
        try:
            response = requests.get(upload_url, headers=HEADERS)
            if response.status_code == 200:
                data = response.json()
                upload_url = data.get("data")
                print(f"✅ Получен специальный URL: {upload_url}")
            else:
                print("❌ Не удалось получить URL для загрузки")
                return
        except Exception as e:
            print(f"❌ Ошибка: {e}")
            return
    else:
        upload_url = f"{BASE_URL}/files"
    
    # Отправляем файл
    try:
        with open(TEST_FILE_NAME, 'rb') as file:
            files = {'file': (TEST_FILE_NAME, file)}
            response = requests.post(upload_url, headers=HEADERS, files=files)
            print_json_response(response)
            
            if response.status_code == 200:
                result = response.json()
                if "data" in result and "id" in result["data"]:
                    analysis_id = result['data']['id']
                    print(f"\n✅ Файл отправлен! ID анализа: {analysis_id}")
                    print(f"   Сохраните этот ID для проверки статуса (функция 10)")
                    
                    # Сохраняем в локальную историю
                    add_to_local_history("analyses", {
                        "type": "file",
                        "name": TEST_FILE_NAME,
                        "hash": file_hash,
                        "analysis_id": analysis_id,
                        "size": file_size,
                        "status": "отправлен"
                    })
                    
                    # Сохраняем ID в переменную для удобства
                    global FILE_ANALYSIS_ID
                    FILE_ANALYSIS_ID = analysis_id
    except Exception as e:
        print(f"❌ Ошибка при отправке файла: {e}")


def function_scan_url():
    """Функция 6: Отправка URL на анализ"""
    print("\n📤 ОТПРАВКА URL НА АНАЛИЗ")
    print(f"Отправляем URL: {TEST_URL}")
    
    url = f"{BASE_URL}/urls"
    data = {"url": TEST_URL}
    
    try:
        response = requests.post(url, headers=HEADERS, data=data)
        print_json_response(response)
        
        if response.status_code == 200:
            result = response.json()
            if "data" in result and "id" in result["data"]:
                analysis_id = result['data']['id']
                print(f"\n✅ URL отправлен! ID анализа: {analysis_id}")
                print(f"   Сохраните этот ID для проверки статуса (функция 10)")
                
                # Сохраняем в локальную историю
                add_to_local_history("analyses", {
                    "type": "url",
                    "url": TEST_URL,
                    "analysis_id": analysis_id,
                    "status": "отправлен"
                })
                
                # Сохраняем ID в переменную для удобства
                global URL_ANALYSIS_ID
                URL_ANALYSIS_ID = analysis_id
    except Exception as e:
        print(f"❌ Ошибка запроса: {e}")


def function_file_comments():
    """Функция 7: Комментарии к файлу"""
    print("\n💬 КОММЕНТАРИИ К ФАЙЛУ")
    
    # Проверяем существование файла
    if not os.path.exists(TEST_FILE_NAME):
        print(f"❌ Файл '{TEST_FILE_NAME}' не найден в текущей директории!")
        return
    
    file_hash = calculate_file_hash(TEST_FILE_NAME)
    if not file_hash:
        return
    
    print(f"📁 Файл: {TEST_FILE_NAME}")
    print(f"🔐 SHA-256: {file_hash}")
    
    url = f"{BASE_URL}/files/{file_hash}/comments"
    
    try:
        response = requests.get(url, headers=HEADERS)
        print_json_response(response)
    except Exception as e:
        print(f"❌ Ошибка запроса: {e}")


def function_file_votes():
    """Функция 8: Голосование по файлу"""
    print("\n🗳️ ГОЛОСОВАНИЕ ПО ФАЙЛУ")
    
    # Проверяем существование файла
    if not os.path.exists(TEST_FILE_NAME):
        print(f"❌ Файл '{TEST_FILE_NAME}' не найден в текущей директории!")
        return
    
    file_hash = calculate_file_hash(TEST_FILE_NAME)
    if not file_hash:
        return
    
    print(f"📁 Файл: {TEST_FILE_NAME}")
    print(f"🔐 SHA-256: {file_hash}")
    
    url = f"{BASE_URL}/files/{file_hash}/votes"
    
    try:
        response = requests.get(url, headers=HEADERS)
        print_json_response(response)
    except Exception as e:
        print(f"❌ Ошибка запроса: {e}")


def function_reanalyse_file():
    """Функция 9: Повторный анализ файла"""
    print("\n🔄 ПОВТОРНЫЙ АНАЛИЗ ФАЙЛА")
    
    # Проверяем существование файла
    if not os.path.exists(TEST_FILE_NAME):
        print(f"❌ Файл '{TEST_FILE_NAME}' не найден в текущей директории!")
        return
    
    file_hash = calculate_file_hash(TEST_FILE_NAME)
    if not file_hash:
        return
    
    print(f"📁 Файл: {TEST_FILE_NAME}")
    print(f"🔐 SHA-256: {file_hash}")
    
    url = f"{BASE_URL}/files/{file_hash}/analyse"
    
    try:
        response = requests.get(url, headers=HEADERS)
        print_json_response(response)
        
        if response.status_code == 200:
            result = response.json()
            if "data" in result and "id" in result["data"]:
                analysis_id = result['data']['id']
                print(f"\n✅ Запрос на повторный анализ отправлен! ID: {analysis_id}")
                
                # Сохраняем в локальную историю
                add_to_local_history("analyses", {
                    "type": "file_reanalysis",
                    "name": TEST_FILE_NAME,
                    "hash": file_hash,
                    "analysis_id": analysis_id,
                    "status": "повторный анализ"
                })
    except Exception as e:
        print(f"❌ Ошибка запроса: {e}")


def function_check_analysis():
    """Функция 10: Проверка статуса анализа по ID"""
    print("\n🔍 ПРОВЕРКА СТАТУСА АНАЛИЗА")
    print("Доступные ID анализов:")
    print(f"  📁 Файл: {FILE_ANALYSIS_ID}")
    if URL_ANALYSIS_ID:
        print(f"  🌐 URL:  {URL_ANALYSIS_ID}")
    else:
        print(f"  🌐 URL:  еще не отправлен")
    
    print("\n1 - Проверить файл")
    print("2 - Проверить URL")
    print("3 - Ввести свой ID")
    print("0 - Назад")
    
    choice = input("\n👉 Выберите: ").strip()
    
    if choice == '1':
        analysis_id = FILE_ANALYSIS_ID
        print(f"\n🔍 Проверяем анализ файла: {analysis_id}")
    elif choice == '2':
        if not URL_ANALYSIS_ID:
            print("❌ URL еще не отправлен на анализ")
            return
        analysis_id = URL_ANALYSIS_ID
        print(f"\n🔍 Проверяем анализ URL: {analysis_id}")
    elif choice == '3':
        analysis_id = input("\nВведите ID анализа: ").strip()
        if not analysis_id:
            print("❌ ID не может быть пустым")
            return
    elif choice == '0':
        return
    else:
        print("❌ Неверный выбор")
        return
    
    url = f"{BASE_URL}/analyses/{analysis_id}"
    
    try:
        response = requests.get(url, headers=HEADERS)
        print_json_response(response)
        
        # Если анализ завершен, показываем статистику
        if response.status_code == 200:
            data = response.json()
            if "data" in data and "attributes" in data["data"]:
                attributes = data["data"]["attributes"]
                if "stats" in attributes:
                    stats = attributes["stats"]
                    print("\n📊 ИТОГОВАЯ СТАТИСТИКА АНАЛИЗА:")
                    print(f"   🟢 Безопасно: {stats.get('harmless', 0)}")
                    print(f"   🟡 Подозрительно: {stats.get('suspicious', 0)}")
                    print(f"   🔴 Вредоносно: {stats.get('malicious', 0)}")
                    print(f"   ⚪ Не обнаружено: {stats.get('undetected', 0)}")
                    
                    if stats.get('malicious', 0) > 0:
                        print("\n⚠️  ВНИМАНИЕ: Обнаружены вредоносные объекты!")
                    elif stats.get('suspicious', 0) > 0:
                        print("\n⚠️  ВНИМАНИЕ: Обнаружены подозрительные объекты!")
                    else:
                        print("\n✅ Результат: Чисто")
    except Exception as e:
        print(f"❌ Ошибка запроса: {e}")


def get_file_relationships(file_hash):
    """Получение связанных объектов для файла"""
    relationships = []
    try:
        # Поведение файла
        url = f"{BASE_URL}/files/{file_hash}/behaviours"
        response = requests.get(url, headers=HEADERS)
        if response.status_code == 200:
            data = response.json()
            if data.get("data"):
                relationships.append(("Поведение", data["data"]))
        
        # Связанные URL
        url = f"{BASE_URL}/files/{file_hash}/contacted_urls"
        response = requests.get(url, headers=HEADERS)
        if response.status_code == 200:
            data = response.json()
            if data.get("data"):
                relationships.append(("Связанные URL", data["data"]))
        
        # Связанные домены
        url = f"{BASE_URL}/files/{file_hash}/contacted_domains"
        response = requests.get(url, headers=HEADERS)
        if response.status_code == 200:
            data = response.json()
            if data.get("data"):
                relationships.append(("Связанные домены", data["data"]))
        
        # Связанные IP
        url = f"{BASE_URL}/files/{file_hash}/contacted_ips"
        response = requests.get(url, headers=HEADERS)
        if response.status_code == 200:
            data = response.json()
            if data.get("data"):
                relationships.append(("Связанные IP", data["data"]))
    except:
        pass
    return relationships


def function_api_history():
    """Функция 11: История активности из API VirusTotal"""
    while True:
        print("\n📊 ИСТОРИЯ АКТИВНОСТИ (из API VirusTotal)")
        print("="*60)
        
        print("\n🔍 Доступные источники истории:")
        print(" 1️⃣  История файла (ELF file)")
        print(" 2️⃣  История URL (rtmis.ru)")
        print(" 3️⃣  История домена (rtmis.ru)")
        print(" 4️⃣  История IP (85.142.23.34)")
        print(" 5️⃣  История анализов")
        print(" 6️⃣  Связанные объекты")
        print(" 7️⃣  Локальная история запросов")
        print(" 8️⃣  Проверить статус ожидающих анализов")
        print(" 0️⃣  Назад")
        
        choice = input("\n👉 Выберите: ").strip()
        
        if choice == '1':
            show_file_history()
        elif choice == '2':
            show_url_history()
        elif choice == '3':
            show_domain_history()
        elif choice == '4':
            show_ip_history()
        elif choice == '5':
            show_analyses_history()
        elif choice == '6':
            show_relationships()
        elif choice == '7':
            show_local_history()
        elif choice == '8':
            show_pending_analyses()
        elif choice == '0':
            break
        else:
            print("❌ Неверный выбор")
        
        input("\n⏎ Нажмите Enter для продолжения...")


def show_file_history():
    """Показать историю файла"""
    print("\n📁 ИСТОРИЯ ФАЙЛА")
    print("="*50)
    
    if not os.path.exists(TEST_FILE_NAME):
        print(f"❌ Файл '{TEST_FILE_NAME}' не найден")
        return
    
    file_hash = calculate_file_hash(TEST_FILE_NAME)
    if not file_hash:
        return
    
    print(f"Файл: {TEST_FILE_NAME}")
    print(f"SHA-256: {file_hash}")
    
    # Получаем информацию о файле
    url = f"{BASE_URL}/files/{file_hash}"
    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code == 200:
            data = response.json()
            attributes = data["data"]["attributes"]
            
            print(f"\n📊 Статистика:")
            print(f"   Первая загрузка: {format_timestamp(attributes.get('first_submission_date'))}")
            print(f"   Последняя загрузка: {format_timestamp(attributes.get('last_submission_date'))}")
            print(f"   Количество загрузок: {attributes.get('times_submitted', 0)}")
            print(f"   Уникальных источников: {attributes.get('unique_sources', 0)}")
            print(f"   Размер: {attributes.get('size', 0)} байт")
            print(f"   Тип: {attributes.get('type_description', 'неизвестно')}")
            
            if "last_analysis_stats" in attributes:
                stats = attributes["last_analysis_stats"]
                print(f"\n🛡️ Результаты последнего анализа:")
                print(f"   🟢 Безопасно: {stats.get('harmless', 0)}")
                print(f"   🟡 Подозрительно: {stats.get('suspicious', 0)}")
                print(f"   🔴 Вредоносно: {stats.get('malicious', 0)}")
                print(f"   ⚪ Не обнаружено: {stats.get('undetected', 0)}")
            
            # Показываем имена файлов
            if "names" in attributes and attributes["names"]:
                print(f"\n📝 Имена файла:")
                for name in attributes["names"][:5]:
                    print(f"   • {name}")
            
            # Показываем комментарии
            comments_url = f"{BASE_URL}/files/{file_hash}/comments"
            comments_response = requests.get(comments_url, headers=HEADERS)
            if comments_response.status_code == 200:
                comments_data = comments_response.json()
                comments = comments_data.get("data", [])
                if comments:
                    print(f"\n💬 Комментарии ({len(comments)}):")
                    for comment in comments[:3]:  # Показываем только первые 3
                        if "attributes" in comment:
                            attr = comment["attributes"]
                            print(f"   • {attr.get('text', '')[:100]}...")
                            print(f"     Автор: {attr.get('user_name', 'неизвестно')}")
                            print(f"     Дата: {format_timestamp(attr.get('date'))}")
            
            # Показываем голосования
            votes_url = f"{BASE_URL}/files/{file_hash}/votes"
            votes_response = requests.get(votes_url, headers=HEADERS)
            if votes_response.status_code == 200:
                votes_data = votes_response.json()
                votes = votes_data.get("data", [])
                if votes:
                    print(f"\n🗳️ Голосования:")
                    malicious = sum(1 for v in votes if v.get("attributes", {}).get("verdict") == "malicious")
                    harmless = sum(1 for v in votes if v.get("attributes", {}).get("verdict") == "harmless")
                    print(f"   Вредоносно: {malicious}")
                    print(f"   Безопасно: {harmless}")
        else:
            print(f"\n❌ Файл не найден в базе VirusTotal")
            print("   Сначала загрузите файл через функцию 5")
    except Exception as e:
        print(f"❌ Ошибка: {e}")


def show_url_history():
    """Показать историю URL"""
    print("\n🌐 ИСТОРИЯ URL")
    print("="*50)
    
    url_id = encode_url_sha256(TEST_URL)
    
    print(f"URL: {TEST_URL}")
    print(f"ID: {url_id}")
    
    # Сначала проверяем, есть ли ожидающий анализ
    if URL_ANALYSIS_ID:
        status, stats, data = check_analysis_status(URL_ANALYSIS_ID)
        print(f"\n⏳ Статус отправленного анализа: {status.upper()}")
        if status == "completed" and stats:
            print(f"   🟢 Безопасно: {stats.get('harmless', 0)}")
            print(f"   🟡 Подозрительно: {stats.get('suspicious', 0)}")
            print(f"   🔴 Вредоносно: {stats.get('malicious', 0)}")
            print(f"   ⚪ Не обнаружено: {stats.get('undetected', 0)}")
    
    # Получаем информацию о URL
    api_url = f"{BASE_URL}/urls/{url_id}"
    try:
        response = requests.get(api_url, headers=HEADERS)
        if response.status_code == 200:
            data = response.json()
            attributes = data["data"]["attributes"]
            
            print(f"\n📊 Статистика:")
            print(f"   Последний анализ: {format_timestamp(attributes.get('last_analysis_date'))}")
            print(f"   Репутация: {attributes.get('reputation', 0)}")
            print(f"   Заголовок: {attributes.get('title', 'неизвестно')}")
            
            if "last_analysis_stats" in attributes:
                stats = attributes["last_analysis_stats"]
                print(f"\n🛡️ Результаты последнего анализа:")
                print(f"   🟢 Безопасно: {stats.get('harmless', 0)}")
                print(f"   🟡 Подозрительно: {stats.get('suspicious', 0)}")
                print(f"   🔴 Вредоносно: {stats.get('malicious', 0)}")
                print(f"   ⚪ Не обнаружено: {stats.get('undetected', 0)}")
            
            # Показываем последние результаты
            if "last_analysis_results" in attributes:
                results = attributes["last_analysis_results"]
                malicious = [r for r in results.values() if r.get("category") == "malicious"]
                if malicious:
                    print(f"\n⚠️ Обнаружены вредоносные движки:")
                    for engine in malicious[:5]:
                        print(f"   • {engine.get('engine_name')}: {engine.get('result')}")
        else:
            print(f"\n⏳ URL еще не проиндексирован в базе VirusTotal")
            print("   Анализ может занять от нескольких секунд до нескольких минут")
            print("   Используйте функцию 10 для проверки статуса анализа")
    except Exception as e:
        print(f"❌ Ошибка: {e}")


def show_domain_history():
    """Показать историю домена"""
    print("\n🏢 ИСТОРИЯ ДОМЕНА")
    print("="*50)
    
    print(f"Домен: {TEST_DOMAIN}")
    
    # Получаем информацию о домене
    api_url = f"{BASE_URL}/domains/{TEST_DOMAIN}"
    try:
        response = requests.get(api_url, headers=HEADERS)
        if response.status_code == 200:
            data = response.json()
            attributes = data["data"]["attributes"]
            
            print(f"\n📊 Статистика:")
            print(f"   Регистратор: {attributes.get('registrar', 'неизвестно')}")
            print(f"   WHOIS обновлен: {format_timestamp(attributes.get('whois_date'))}")
            print(f"   Репутация: {attributes.get('reputation', 0)}")
            
            if "last_analysis_stats" in attributes:
                stats = attributes["last_analysis_stats"]
                print(f"\n🛡️ Результаты анализа:")
                print(f"   🟢 Безопасно: {stats.get('harmless', 0)}")
                print(f"   🟡 Подозрительно: {stats.get('suspicious', 0)}")
                print(f"   🔴 Вредоносно: {stats.get('malicious', 0)}")
                print(f"   ⚪ Не обнаружено: {stats.get('undetected', 0)}")
            
            # Показываем DNS записи
            if "last_dns_records" in attributes:
                print(f"\n📡 Последние DNS записи:")
                for record in attributes["last_dns_records"][:5]:
                    record_type = record.get("type", "Unknown")
                    value = record.get("value", "")
                    print(f"   • {record_type}: {value}")
            
            # Показываем категории
            if "categories" in attributes and attributes["categories"]:
                print(f"\n📂 Категории:")
                for category in list(attributes["categories"].items())[:5]:
                    print(f"   • {category[0]}: {category[1]}")
        else:
            print(f"\n❌ Домен не найден в базе VirusTotal")
    except Exception as e:
        print(f"❌ Ошибка: {e}")


def show_ip_history():
    """Показать историю IP"""
    print("\n📍 ИСТОРИЯ IP-АДРЕСА")
    print("="*50)
    
    print(f"IP: {TEST_IP}")
    
    # Получаем информацию о IP
    api_url = f"{BASE_URL}/ip_addresses/{TEST_IP}"
    try:
        response = requests.get(api_url, headers=HEADERS)
        if response.status_code == 200:
            data = response.json()
            attributes = data["data"]["attributes"]
            
            print(f"\n📊 Информация:")
            print(f"   Страна: {attributes.get('country', 'неизвестно')}")
            print(f"   ASN: {attributes.get('asn', 'неизвестно')}")
            print(f"   Владелец: {attributes.get('as_owner', 'неизвестно')}")
            print(f"   Сеть: {attributes.get('network', 'неизвестно')}")
            
            if "last_analysis_stats" in attributes:
                stats = attributes["last_analysis_stats"]
                print(f"\n🛡️ Результаты анализа:")
                print(f"   🟢 Безопасно: {stats.get('harmless', 0)}")
                print(f"   🟡 Подозрительно: {stats.get('suspicious', 0)}")
                print(f"   🔴 Вредоносно: {stats.get('malicious', 0)}")
                print(f"   ⚪ Не обнаружено: {stats.get('undetected', 0)}")
            
            # WHOIS информация
            if "whois" in attributes:
                print(f"\n📝 WHOIS:")
                whois_lines = attributes["whois"].split('\n')[:5]
                for line in whois_lines:
                    print(f"   {line}")
        else:
            print(f"\n❌ IP не найден в базе VirusTotal")
    except Exception as e:
        print(f"❌ Ошибка: {e}")


def show_analyses_history():
    """Показать историю анализов"""
    print("\n🔬 ИСТОРИЯ АНАЛИЗОВ")
    print("="*50)
    
    # Проверяем файл
    if os.path.exists(TEST_FILE_NAME):
        file_hash = calculate_file_hash(TEST_FILE_NAME)
        if file_hash:
            print(f"\n📁 Файл: {TEST_FILE_NAME}")
            url = f"{BASE_URL}/files/{file_hash}"
            try:
                response = requests.get(url, headers=HEADERS)
                if response.status_code == 200:
                    data = response.json()
                    attributes = data["data"]["attributes"]
                    print(f"   Загружен: {format_timestamp(attributes.get('first_submission_date'))}")
                    print(f"   Размер: {attributes.get('size', 0)} байт")
                    print(f"   Тип: {attributes.get('type_description', 'неизвестно')}")
                    
                    if "last_analysis_stats" in attributes:
                        stats = attributes["last_analysis_stats"]
                        print(f"   Результат: 🟢{stats.get('harmless',0)} 🟡{stats.get('suspicious',0)} 🔴{stats.get('malicious',0)}")
                else:
                    print(f"   Статус: Файл загружен, ожидает индексации")
            except:
                pass
    
    # Проверяем URL
    print(f"\n🌐 URL: {TEST_URL}")
    if URL_ANALYSIS_ID:
        status, stats, data = check_analysis_status(URL_ANALYSIS_ID)
        print(f"   Статус анализа: {status.upper()}")
        if stats:
            print(f"   Результат: 🟢{stats.get('harmless',0)} 🟡{stats.get('suspicious',0)} 🔴{stats.get('malicious',0)}")
    else:
        print("   URL не отправлен на анализ")


def show_relationships():
    """Показать связанные объекты"""
    print("\n🔗 СВЯЗАННЫЕ ОБЪЕКТЫ")
    print("="*50)
    
    if not os.path.exists(TEST_FILE_NAME):
        print(f"❌ Файл '{TEST_FILE_NAME}' не найден")
        return
    
    file_hash = calculate_file_hash(TEST_FILE_NAME)
    if not file_hash:
        return
    
    print(f"\n📁 Для файла: {TEST_FILE_NAME}")
    print(f"SHA-256: {file_hash}")
    
    relationships = get_file_relationships(file_hash)
    
    if relationships:
        for rel_type, rel_data in relationships:
            print(f"\n{rel_type}:")
            if isinstance(rel_data, list):
                for item in rel_data[:3]:  # Показываем только первые 3
                    print(f"   • {item}")
    else:
        print("\n   Нет связанных объектов или они еще не определены")


def show_local_history():
    """Показать локальную историю запросов"""
    print("\n📋 ЛОКАЛЬНАЯ ИСТОРИЯ ЗАПРОСОВ")
    print("="*50)
    
    history = load_local_history()
    
    total_requests = sum(len(history.get(cat, [])) for cat in history)
    print(f"\n📊 Всего запросов: {total_requests}")
    
    categories = [
        ("file_checks", "📁 Проверки файлов"),
        ("url_checks", "🌐 Проверки URL"),
        ("domain_checks", "🏢 Проверки доменов"),
        ("ip_checks", "📍 Проверки IP"),
        ("analyses", "🔬 Анализы")
    ]
    
    for cat_key, cat_name in categories:
        items = history.get(cat_key, [])
        if items:
            print(f"\n{cat_name} ({len(items)}):")
            for item in reversed(items[-5:]):  # Последние 5
                print(f"   • {item['date']}")
                data = item['data']
                if cat_key == "file_checks":
                    print(f"     Файл: {data.get('name', 'неизвестно')}")
                    print(f"     Результат: {data.get('result', 'неизвестно')}")
                elif cat_key == "url_checks":
                    print(f"     URL: {data.get('url', 'неизвестно')}")
                    print(f"     Результат: {data.get('result', 'неизвестно')}")
                elif cat_key == "domain_checks":
                    print(f"     Домен: {data.get('domain', 'неизвестно')}")
                    print(f"     Результат: {data.get('result', 'неизвестно')}")
                elif cat_key == "ip_checks":
                    print(f"     IP: {data.get('ip', 'неизвестно')}")
                    print(f"     Результат: {data.get('result', 'неизвестно')}")
                elif cat_key == "analyses":
                    print(f"     Тип: {data.get('type', 'неизвестно')}")
                    print(f"     Статус: {data.get('status', 'неизвестно')}")


def show_pending_analyses():
    """Показать статус ожидающих анализов"""
    print("\n⏳ ОЖИДАЮЩИЕ АНАЛИЗЫ")
    print("="*50)
    
    if FILE_ANALYSIS_ID:
        print(f"\n📁 Файл: {TEST_FILE_NAME}")
        status, stats, data = check_analysis_status(FILE_ANALYSIS_ID)
        print(f"   Статус: {status.upper()}")
        print(f"   ID: {FILE_ANALYSIS_ID}")
        if stats:
            print(f"   Результат: 🟢{stats.get('harmless',0)} 🟡{stats.get('suspicious',0)} 🔴{stats.get('malicious',0)}")
    
    if URL_ANALYSIS_ID:
        print(f"\n🌐 URL: {TEST_URL}")
        status, stats, data = check_analysis_status(URL_ANALYSIS_ID)
        print(f"   Статус: {status.upper()}")
        print(f"   ID: {URL_ANALYSIS_ID}")
        if stats:
            print(f"   Результат: 🟢{stats.get('harmless',0)} 🟡{stats.get('suspicious',0)} 🔴{stats.get('malicious',0)}")
    
    if not FILE_ANALYSIS_ID and not URL_ANALYSIS_ID:
        print("\n   Нет ожидающих анализов")


def main():
    """Основная функция программы"""
    while True:
        print_header()
        print_menu()
        
        choice = input("\n👉 Выберите функцию (0-11): ").strip()
        
        if choice == '0':
            print("\n👋 Выход из программы. До свидания!")
            sys.exit(0)
        elif choice == '1':
            function_file_info()
        elif choice == '2':
            function_url_info()
        elif choice == '3':
            function_domain_info()
        elif choice == '4':
            function_ip_info()
        elif choice == '5':
            function_upload_file()
        elif choice == '6':
            function_scan_url()
        elif choice == '7':
            function_file_comments()
        elif choice == '8':
            function_file_votes()
        elif choice == '9':
            function_reanalyse_file()
        elif choice == '10':
            function_check_analysis()
        elif choice == '11':
            function_api_history()
        else:
            print("\n❌ Неверный выбор! Пожалуйста, выберите число от 0 до 11.")
        
        input("\n⏎ Нажмите Enter для продолжения...")


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n👋 Программа прервана пользователем. До свидания!")
        sys.exit(0)
    except Exception as e:
        print(f"\n❌ Непредвиденная ошибка: {e}")
        sys.exit(1)


🐛 ЛАБОРАТОРНАЯ РАБОТА: ВЗАИМОДЕЙСТВИЕ С API VIRUSTOTAL
🔑 API Key: c7cce2f13b...d3bf1b5f9e
📁 Файл для анализа: ELF file
🌐 Сайт для проверки: https://rtmis.ru

📋 ДОСТУПНЫЕ ФУНКЦИИ:
 1️⃣  Информация о файле (ELF file)
 2️⃣  Информация о сайте (rtmis.ru)
 3️⃣  Информация о домене (rtmis.ru)
 4️⃣  Информация об IP-адресе (rtmis.ru)
 5️⃣  Отправить файл на анализ
 6️⃣  Отправить URL на анализ
 7️⃣  Комментарии к файлу
 8️⃣  Голосование по файлу
 9️⃣  Повторный анализ файла
 🔟  Проверить статус анализа (по ID)
 1️⃣1️⃣  История активности (из API VirusTotal)
 0️⃣  Выход
------------------------------------------------------------



👉 Выберите функцию (0-11):  1



🔍 ИНФОРМАЦИЯ О ФАЙЛЕ
📁 Файл: ELF file
🔐 SHA-256: 2b87e52580a8336b7d951ef8cbc5b4f996405fcfe739d557c378af76eb1e9354

✅ Статус: 200 OK
📦 ОТВЕТ ОТ СЕРВЕРА:
{
  "data": {
    "id": "2b87e52580a8336b7d951ef8cbc5b4f996405fcfe739d557c378af76eb1e9354",
    "type": "file",
    "links": {
      "self": "https://www.virustotal.com/api/v3/files/2b87e52580a8336b7d951ef8cbc5b4f996405fcfe739d557c378af76eb1e9354"
    },
    "attributes": {
      "detectiteasy": {
        "filetype": "ELF64",
        "values": [
          {
            "info": "DYN AMD64-64",
            "type": "Operation system",
            "name": "Unix"
          },
          {
            "info": "DYN AMD64-64",
            "version": "2.4",
            "type": "Library",
            "name": "GLIBC"
          },
          {
            "info": "DYN AMD64-64",
            "version": "(GNU) 15.2.1 20260209",
            "type": "Compiler",
            "name": "gcc"
          }
        ]
      },
      "sandbox_verdicts": {
        


⏎ Нажмите Enter для продолжения... 



🐛 ЛАБОРАТОРНАЯ РАБОТА: ВЗАИМОДЕЙСТВИЕ С API VIRUSTOTAL
🔑 API Key: c7cce2f13b...d3bf1b5f9e
📁 Файл для анализа: ELF file
🌐 Сайт для проверки: https://rtmis.ru

📋 ДОСТУПНЫЕ ФУНКЦИИ:
 1️⃣  Информация о файле (ELF file)
 2️⃣  Информация о сайте (rtmis.ru)
 3️⃣  Информация о домене (rtmis.ru)
 4️⃣  Информация об IP-адресе (rtmis.ru)
 5️⃣  Отправить файл на анализ
 6️⃣  Отправить URL на анализ
 7️⃣  Комментарии к файлу
 8️⃣  Голосование по файлу
 9️⃣  Повторный анализ файла
 🔟  Проверить статус анализа (по ID)
 1️⃣1️⃣  История активности (из API VirusTotal)
 0️⃣  Выход
------------------------------------------------------------


...


👉 Выберите:  0

⏎ Нажмите Enter для продолжения... 



🐛 ЛАБОРАТОРНАЯ РАБОТА: ВЗАИМОДЕЙСТВИЕ С API VIRUSTOTAL
🔑 API Key: c7cce2f13b...d3bf1b5f9e
📁 Файл для анализа: ELF file
🌐 Сайт для проверки: https://rtmis.ru

📋 ДОСТУПНЫЕ ФУНКЦИИ:
 1️⃣  Информация о файле (ELF file)
 2️⃣  Информация о сайте (rtmis.ru)
 3️⃣  Информация о домене (rtmis.ru)
 4️⃣  Информация об IP-адресе (rtmis.ru)
 5️⃣  Отправить файл на анализ
 6️⃣  Отправить URL на анализ
 7️⃣  Комментарии к файлу
 8️⃣  Голосование по файлу
 9️⃣  Повторный анализ файла
 🔟  Проверить статус анализа (по ID)
 1️⃣1️⃣  История активности (из API VirusTotal)
 0️⃣  Выход
------------------------------------------------------------



👉 Выберите функцию (0-11):  0



👋 Выход из программы. До свидания!


SystemExit: 0